In [1]:
import pandas as pd
import glob
files = glob.glob("D:/Project/Dodge/sap-o2c-data/payments_accounts_receivable/*.jsonl")

df_pay = pd.concat(
    [pd.read_json(f, lines=True) for f in files],
    ignore_index=True
)

print(df_pay.columns)

Index(['companyCode', 'fiscalYear', 'accountingDocument',
       'accountingDocumentItem', 'clearingDate', 'clearingAccountingDocument',
       'clearingDocFiscalYear', 'amountInTransactionCurrency',
       'transactionCurrency', 'amountInCompanyCodeCurrency',
       'companyCodeCurrency', 'customer', 'invoiceReference',
       'invoiceReferenceFiscalYear', 'salesDocument', 'salesDocumentItem',
       'postingDate', 'documentDate', 'assignmentReference', 'glAccount',
       'financialAccountType', 'profitCenter', 'costCenter'],
      dtype='object')


In [2]:
df_pay = df_pay.rename(columns={
    "accountingDocument": "payment_id",
    "customer": "customer_id",
    "amountInTransactionCurrency": "amount",
    "invoiceReference": "invoice_id",
    "postingDate": "payment_date"
})

In [3]:
df_pay = df_pay[[
    "payment_id",
    "customer_id",
    "invoice_id",
    "amount",
    "payment_date"
]]

In [4]:
import sqlite3

conn = sqlite3.connect("data.db")
df_pay.to_sql("payments", conn, if_exists="replace", index=False)

print("✅ payments table created")

✅ payments table created


In [5]:
query = """
SELECT 
    i.invoice_id,
    i.customer_id,
    ii.product_id,
    ii.delivery_id,
    d.shipping_point,
    p.payment_id,
    p.amount as payment_amount
FROM invoices i
JOIN invoice_items ii ON i.invoice_id = ii.invoice_id
LEFT JOIN deliveries d ON ii.delivery_id = d.delivery_id
LEFT JOIN payments p ON i.invoice_id = p.invoice_id
LIMIT 5;
"""

print(pd.read_sql(query, conn))

   invoice_id  customer_id      product_id  delivery_id shipping_point  \
0    90504248    320000083  S8907367001003     80738072           WB05   
1    90628265    320000082  S8907367013532     80754575           1301   
2    90628266    320000082  S8907367002512     80754576           1301   
3    90504274    320000083  S8907367003380     80738091           WB05   
4    90504242    320000083  S8907367025344     80738068           WB05   

  payment_id payment_amount  
0       None           None  
1       None           None  
2       None           None  
3       None           None  
4       None           None  


In [6]:

query = """
SELECT product_id, COUNT(*) as total
FROM invoice_items
GROUP BY product_id
ORDER BY total DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,product_id,total
0,S8907367039280,22
1,S8907367008620,22
2,S8907367042006,16
3,S8907367013532,15
4,S8907367001003,9


In [7]:
query = """
SELECT product_id, COUNT(*) as total
FROM invoice_items
GROUP BY product_id
ORDER BY total DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,product_id,total
0,S8907367039280,22
1,S8907367008620,22
2,S8907367042006,16
3,S8907367013532,15
4,S8907367001003,9


In [8]:
query = """
SELECT 
    i.invoice_id,
    i.customer_id,
    ii.product_id,
    ii.delivery_id,
    d.shipping_point,
    p.payment_id
FROM invoices i
JOIN invoice_items ii ON i.invoice_id = ii.invoice_id
LEFT JOIN deliveries d ON ii.delivery_id = d.delivery_id
LEFT JOIN payments p ON i.invoice_id = p.invoice_id
WHERE i.invoice_id = '90504248';
"""

pd.read_sql(query, conn)

,invoice_id,customer_id,product_id,delivery_id,shipping_point,payment_id
0,90504248,320000083,S8907367001003,80738072,WB05,None


In [9]:
query = """
SELECT 
    i.invoice_id,
    i.customer_id,
    ii.product_id,
    ii.delivery_id,
    d.shipping_point,
    p.payment_id
FROM invoices i
JOIN invoice_items ii ON i.invoice_id = ii.invoice_id
LEFT JOIN deliveries d ON ii.delivery_id = d.delivery_id
LEFT JOIN payments p ON i.invoice_id = p.invoice_id
WHERE i.invoice_id = '90504248';
"""

pd.read_sql(query, conn)

,invoice_id,customer_id,product_id,delivery_id,shipping_point,payment_id
0,90504248,320000083,S8907367001003,80738072,WB05,None


In [10]:
query = """
SELECT d.delivery_id
FROM deliveries d
LEFT JOIN invoice_items ii 
ON d.delivery_id = ii.delivery_id
WHERE ii.delivery_id IS NULL;
"""

pd.read_sql(query, conn)

,delivery_id
0,80737721
1,80737722
2,80737723


In [11]:
query = """
SELECT i.invoice_id
FROM invoices i
LEFT JOIN payments p 
ON i.invoice_id = p.invoice_id
WHERE p.invoice_id IS NULL;
"""

pd.read_sql(query, conn)

,invoice_id
0,90504248
1,90628265
2,90628266
3,90504274
4,90504242
...,...
158,91150213
159,91150214
160,91150215
161,91150216
